Testing de algunos modelos

In [9]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, make_scorer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor, Pool
import shap
import lightgbm as lgb
from lightgbm import Dataset
from xgboost import XGBRegressor
import pandas as pd
from sklearn.model_selection import KFold, RandomizedSearchCV
import numpy as np
import pandas as pd

In [10]:
df_prep = pd.read_csv('../data/processed/BC_A&A_with_ATD_prep.csv', index_col=0)

In [18]:
target = 'ATD_log'
X = df_prep.drop(columns=[target]).copy()
y = df_prep[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

cv = KFold(n_splits=3, shuffle=True, random_state=42)
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

In [ ]:
search_space = {

    # A) XGBRegressor  ───────────────────────────────────────────────────────
    'xgb': {
        'estimator': XGBRegressor(
            objective='reg:absoluteerror',
            tree_method='hist',
            n_jobs=-1,
            random_state=42,
            eval_metric='mae'),
        'param_distributions': {
            'n_estimators'      : [400, 600, 900, 1200],
            'learning_rate'     : [0.01, 0.03, 0.05, 0.1],
            'max_depth'         : [4, 6, 8, 10],
            'min_child_weight'  : [1, 5, 10],
            'subsample'         : [0.6, 0.8, 1.0],
            'colsample_bytree'  : [0.6, 0.8, 1.0],
            'reg_alpha'         : [0.0, 0.1, 0.5, 1.0],
            'reg_lambda'        : [1.0, 2.0, 5.0]
        },
        'n_iter': 1
    },

    # B) LightGBM ────────────────────────────────────────────────────────────
    'lgb': {
        'estimator': LGBMRegressor(
            objective='mae',
            boosting_type='gbdt',
            n_jobs=-1,
            random_state=42),
        'param_distributions': {
            'n_estimators'      : [600, 900, 1200, 1500],
            'learning_rate'     : [0.01, 0.03, 0.05, 0.1],
            'num_leaves'        : [31, 63, 127, 255],
            'max_depth'         : [-1, 8, 12, 16],
            'min_child_samples' : [10, 20, 40, 80],
            'subsample'         : [0.6, 0.8, 1.0],
            'colsample_bytree'  : [0.6, 0.8, 1.0],
            'reg_alpha'         : [0.0, 0.1, 0.5, 1.0],
            'reg_lambda'        : [0.0, 0.1, 0.5, 1.0]
        },
        'n_iter': 1
    },

    # C) CatBoost ────────────────────────────────────────────────────────────
    'cat': {
        'estimator': CatBoostRegressor(
            loss_function='MAE',
            thread_count=-1,
            random_state=42,
            verbose=False),
        'param_distributions': {
            'iterations'        : [600, 900, 1200],
            'learning_rate'     : [0.02, 0.05, 0.1],
            'depth'             : [4, 6, 8, 10],
            'l2_leaf_reg'       : [1, 3, 5, 7, 9],
            'bagging_temperature': [0.0, 0.5, 1.0],
            'subsample'         : [0.6, 0.8, 1.0]
        },
        'n_iter': 1
    }
}

results = {}

for label, cfg in search_space.items():
    print(f"\n── Optimizing {label.upper()}  ({cfg['n_iter']} iterations) ──")
    rs = RandomizedSearchCV(
            estimator=cfg['estimator'],
            param_distributions=cfg['param_distributions'],
            n_iter=cfg['n_iter'],
            scoring=mae_scorer,
            cv=cv,
            n_jobs=-1,
            verbose=1,
            random_state=42
        )
    rs.fit(X_train, y_train)
    best = rs.best_estimator_
    results[label] = best
    print(f"   • Best CV MAE = {-rs.best_score_:.4f}")
    print(f"   • Params      = {rs.best_params_}")

# ─────────────────── 3. EVALUACIÓN EN HOLD-OUT ─────────────────────────────
print("\n────────  Final performance on untouched test set  ────────")
for label, model in results.items():
    y_pred = model.predict(X_test)
    mae    = mean_absolute_error(y_test, y_pred)
    print(f"{label.upper():4s} | MAE_test = {mae:.4f}")


── Optimizing XGB  (1 iterations) ──
Fitting 3 folds for each of 1 candidates, totalling 3 fits
   • Best CV MAE = 0.2838
   • Params      = {'subsample': 0.6, 'reg_lambda': 1.0, 'reg_alpha': 1.0, 'n_estimators': 900, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.03, 'colsample_bytree': 1.0}

── Optimizing LGB  (1 iterations) ──
Fitting 3 folds for each of 1 candidates, totalling 3 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.029350 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 881
[LightGBM] [Info] Number of data points in the train set: 533333, number of used features: 40
[Lig